In [ ]:
print("=" * 60)
print("NEURAL NETWORK - Learning Curve")
print("=" * 60)

# Use sklearn's MLPClassifier for neural network
from sklearn.neural_network import MLPClassifier

# Neural Network learning curves using sklearn
nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=32,
    learning_rate='constant',
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10
)

nn_curves = plot_learning_curves(
    nn_model,
    X_combined.values,
    y_combined.values,
    'Neural Network',
    model_type='sklearn',
    test_X=X_test.values,
    test_y=y_test.values
)

print("\nNeural Network learning curve computed")

In [ ]:
# Collect all learning curves for analysis
all_curves = [lr_curves, rf_curves, nn_curves]
print("All learning curves collected for analysis")

## 8. Visualize Results

Create comprehensive visualizations comparing the learning curves for all three models.

In [ ]:
# Create summary DataFrame
summary_data = []
for curve in all_curves:
    summary_data.append({
        'Model': curve['model_name'],
        'Train Score': f"{curve['train_mean'][-1]:.4f}",
        'Val Score': f"{curve['val_mean'][-1]:.4f}",
        'Gap': f"{(curve['train_mean'][-1] - curve['val_mean'][-1]):.4f}",
        'Test Score': f"{curve.get('test_score', 'N/A')}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 80)
print("FINAL SUMMARY TABLE")
print("=" * 80)
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv('results/learning_curves_summary.csv', index=False)
print("\n✓ Saved: results/learning_curves_summary.csv")

In [ ]:
# Create additional detailed comparison visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Training vs Validation gap for each model
ax1 = axes[0]
model_names = [c['model_name'] for c in all_curves]
gaps = []

for curve in all_curves:
    gap = curve['train_mean'][-1] - curve['val_mean'][-1]
    gaps.append(gap)

bars = ax1.bar(model_names, gaps, color=['#2E86AB', '#A23B72', '#F18F01'], 
               edgecolor='black', linewidth=1.5, alpha=0.8)
ax1.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='Perfect Generalization')
ax1.set_ylabel('Training - Validation Gap', fontsize=11, fontweight='bold')
ax1.set_title('Overfitting Analysis\n(Higher gap = More overfitting)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, gap in zip(bars, gaps):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{gap:.4f}', ha='center', va='bottom' if height > 0 else 'top', 
            fontweight='bold', fontsize=10)

# Plot 2: Final validation scores comparison
ax2 = axes[1]
final_val_scores = [c['val_mean'][-1] for c in all_curves]

bars2 = ax2.bar(model_names, final_val_scores, color=['#2E86AB', '#A23B72', '#F18F01'],
                edgecolor='black', linewidth=1.5, alpha=0.8)
ax2.set_ylabel('Validation F1 Score', fontsize=11, fontweight='bold')
ax2.set_title('Final Model Performance\n(Using Full Training Data)', fontsize=12, fontweight='bold')
ax2.set_ylim([0, 1.0])
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, score in zip(bars2, final_val_scores):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('results/learning_curves_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: results/learning_curves_analysis.png")
plt.show()

print("\nAdditional analysis plots generated!")

In [ ]:
print("\nGenerating visualizations...")

# Create figure with subplots: one for each model + one comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

models_curves = [
    (lr_curves, 'Logistic Regression', axes[0, 0]),
    (rf_curves, 'Random Forest', axes[0, 1]),
    (nn_curves, 'Neural Network', axes[1, 0])
]

# Plot individual learning curves for each model
for curve, title, ax in models_curves:
    train_sizes = curve['train_sizes']
    train_mean = curve['train_mean']
    train_std = curve['train_std']
    val_mean = curve['val_mean']
    val_std = curve['val_std']
    
    # Plot training scores
    ax.plot(train_sizes, train_mean, 'o-', color='#2E86AB', linewidth=2.5, 
            markersize=6, label='Training Score')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                     alpha=0.2, color='#2E86AB')
    
    # Plot validation scores
    ax.plot(train_sizes, val_mean, 's-', color='#A23B72', linewidth=2.5, 
            markersize=6, label='Validation Score')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                     alpha=0.2, color='#A23B72')
    
    ax.set_xlabel('Training Set Size', fontsize=11, fontweight='bold')
    ax.set_ylabel('F1 Score (Weighted)', fontsize=11, fontweight='bold')
    ax.set_title(f'{title} Learning Curve', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.3, 1.0])

# Comparison plot: Validation curves for all models
ax_comp = axes[1, 1]

colors = ['#2E86AB', '#A23B72', '#F18F01']
for idx, curve in enumerate(all_curves):
    train_sizes = curve['train_sizes']
    val_mean = curve['val_mean']
    ax_comp.plot(train_sizes, val_mean, 'o-', color=colors[idx], linewidth=2.5, 
                markersize=7, label=curve['model_name'])

ax_comp.set_xlabel('Training Set Size', fontsize=11, fontweight='bold')
ax_comp.set_ylabel('Validation F1 Score (Weighted)', fontsize=11, fontweight='bold')
ax_comp.set_title('Model Comparison: Validation Curves', fontsize=12, fontweight='bold')
ax_comp.legend(loc='lower right', fontsize=10)
ax_comp.grid(True, alpha=0.3)
ax_comp.set_ylim([0.3, 1.0])

plt.tight_layout()
plt.savefig('results/learning_curves_all_models.png', dpi=300, bbox_inches='tight')
print("✓ Saved: results/learning_curves_all_models.png")
plt.show()

print("\nVisualization complete!")

## 7. Compare Learning Curves Across Models

Summarize and compare the performance metrics for all three models.

In [ ]:
print("\n" + "=" * 80)
print("LEARNING CURVE COMPARISON SUMMARY")
print("=" * 80)

# Compare metrics at different training sizes
print("\nF1 Scores at Different Training Set Sizes:")
print("-" * 80)
print(f"{'Training Size':<20} {'Logistic Reg (Val)':<25} {'Random Forest (Val)':<25} {'Neural Network (Val)':<25}")
print("-" * 80)

for i, size in enumerate(lr_curves['train_sizes']):
    lr_val = lr_curves['val_mean'][i]
    rf_val = rf_curves['val_mean'][i]
    nn_val = nn_curves['val_mean'][i]
    print(f"{size:<20} {lr_val:<25.4f} {rf_val:<25.4f} {nn_val:<25.4f}")

print("\n" + "=" * 80)
print("Final Test Set Performance (Full Training Data):")
print("=" * 80)
print(f"Logistic Regression:  F1 = {lr_curves.get('test_score', 'N/A')}")
print(f"Random Forest:        F1 = {rf_curves.get('test_score', 'N/A')}")
print(f"Neural Network:       F1 = {nn_curves.get('test_score', 'N/A')}")

print("\n" + "=" * 80)
print("Model Generalization Analysis:")
print("=" * 80)

for curve in all_curves:
    name = curve['model_name']
    train_final = curve['train_mean'][-1]
    val_final = curve['val_mean'][-1]
    gap = train_final - val_final
    
    print(f"\n{name}:")
    print(f"  Training Score:  {train_final:.4f}")
    print(f"  Validation Score: {val_final:.4f}")
    print(f"  Train-Val Gap:   {gap:.4f}")
    
    if gap > 0.1:
        print(f"  → Indicates OVERFITTING (model fits training data better)")
    elif gap < -0.05:
        print(f"  → Indicates UNDERFITTING (model struggles on both)")
    else:
        print(f"  → Good GENERALIZATION (balanced performance)")

## 6. Generate Learning Curves for Neural Network

Train Neural Network using custom PyTorch implementation across different training set sizes.

In [13]:
print("=" * 60)
print("RANDOM FOREST - Learning Curve")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_curves = plot_learning_curves(
    rf_model,
    X_combined_rf.values,
    y_combined_rf.values,
    'Random Forest',
    model_type='sklearn',
    test_X=X_test_rf.values,
    test_y=y_test_rf.values
)

print("\nRandom Forest learning curve computed")

RANDOM FOREST - Learning Curve
[learning_curve] Training set sizes: [ 14249  28499  42749  56998  71248  85498  99747 113997 128247 142497]


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
/usr/local/python/3.12.1/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:  4.0min finished



Random Forest learning curve computed


## 5. Generate Learning Curves for Random Forest

Train Random Forest with optimal hyperparameters across different training set sizes.

In [12]:
print("=" * 60)
print("LOGISTIC REGRESSION - Learning Curve")
print("=" * 60)

lr_model = LogisticRegression(
    C=0.01,
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)

lr_curves = plot_learning_curves(
    lr_model, 
    X_combined.values, 
    y_combined.values, 
    'Logistic Regression',
    model_type='sklearn'
)

print("\nLogistic Regression learning curve computed")

LOGISTIC REGRESSION - Learning Curve
[learning_curve] Training set sizes: [ 14249  28499  42749  56998  71248  85498  99747 113997 128247 142497]


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:   48.8s finished



Logistic Regression learning curve computed


## 4. Generate Learning Curves for Logistic Regression

Train Logistic Regression with the optimal hyperparameter (C=0.01) across different training set sizes.

In [ ]:
print("Learning curve function defined")

## 3. Define Learning Curve Function

Create a helper function to compute learning curves using cross-validation. This function trains models with increasingly larger subsets of the training data and evaluates performance.

In [9]:
# Prepare features for each model type
def prepare_features(df, one_hot_race=False):
    """Prepare features and target from a dataframe."""
    X = df.drop(TARGET, axis=1)
    y = df[TARGET]
    
    if one_hot_race:
        # One-hot encode Race for Logistic Regression and Neural Network
        if 'Race' in X.columns:
            X = pd.get_dummies(X, columns=['Race'], drop_first=True)
    
    return X, y

# Prepare combined train+val for consistent learning curves
train_val = pd.concat([train, val], ignore_index=True)
X_combined, y_combined = prepare_features(train_val, one_hot_race=True)
X_combined_rf, y_combined_rf = prepare_features(train_val, one_hot_race=False)

print(f"\nCombined training data (train+val): {X_combined.shape[0]:,} rows")
print(f"Features (one-hot encoded): {X_combined.shape[1]}")
print(f"Features (integer encoded for RF): {X_combined_rf.shape[1]}")

# Prepare test data
X_test, y_test = prepare_features(test, one_hot_race=True)
X_test_rf, y_test_rf = prepare_features(test, one_hot_race=False)


Combined training data (train+val): 213,746 rows
Features (one-hot encoded): 28
Features (integer encoded for RF): 22


In [8]:
# Load the preprocessed data splits
train = pd.read_csv('data/processed/train.csv')
val = pd.read_csv('data/processed/val.csv')
test = pd.read_csv('data/processed/test.csv')

print(f"Training set: {train.shape[0]:,} rows, {train.shape[1]} columns")
print(f"Validation set: {val.shape[0]:,} rows")
print(f"Test set: {test.shape[0]:,} rows")

TARGET = 'Diabetes_binary'
print(f"\nTarget variable: {TARGET}")
print(f"Number of features: {train.shape[1] - 1}")

# Check class distribution
train_pos_rate = train[TARGET].mean()
print(f"\nClass distribution (positive = diabetes/prediabetes):")
print(f"  Train: {train_pos_rate:.1%} positive ({train[TARGET].sum():,} / {len(train):,})")
print(f"  Imbalance ratio: {(1-train_pos_rate)/train_pos_rate:.1f}:1 (negative:positive)")

Training set: 176,026 rows, 23 columns
Validation set: 37,720 rows
Test set: 37,721 rows

Target variable: Diabetes_binary
Number of features: 22

Class distribution (positive = diabetes/prediabetes):
  Train: 15.7% positive (27,697.0 / 176,026)
  Imbalance ratio: 5.4:1 (negative:positive)


## 2. Load and Prepare Data

Load the preprocessed train/val/test splits and prepare features for model training.

In [11]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

def plot_learning_curves(model, X, y, model_name, model_type='sklearn', test_X=None, test_y=None):
    """
    Compute and return learning curves using cross-validation.
    
    Args:
        model: Sklearn model or 'nn' for neural network
        X: Training features
        y: Training labels
        model_name: Name of the model for display
        model_type: 'sklearn' or 'neural_network'
        test_X: Test features (for final evaluation)
        test_y: Test labels (for final evaluation)
    
    Returns:
        Dictionary with train_sizes and corresponding train/val scores
    """
    
    if model_type == 'sklearn':
        # Use scikit-learn's learning_curve function
        train_sizes = np.linspace(0.1, 1.0, 10)  # 10 - 100% of data
        
        train_sizes, train_scores, val_scores = learning_curve(
            model, X, y,
            train_sizes=train_sizes,
            cv=3,  # 3-fold cross-validation
            scoring='f1_weighted',
            n_jobs=-1,
            verbose=1
        )
        
        # Convert to actual sample sizes
        train_sample_sizes = (train_sizes * len(X)).astype(int)
        
        # Calculate mean and std of scores across folds
        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        val_mean = np.mean(val_scores, axis=1)
        val_std = np.std(val_scores, axis=1)
        
        result = {
            'model_name': model_name,
            'train_sizes': train_sample_sizes,
            'train_mean': train_mean,
            'train_std': train_std,
            'val_mean': val_mean,
            'val_std': val_std
        }
        
    # Add test set performance if provided
    # if test_X is not None and test_y is not None:
        model.fit(X, y)
    #     model.fit(X, y)
    #     test_pred = model.predict(test_X)
    #     test_score = f1_score(test_y, test_pred, average='weighted', zero_division=0)
    #     result['test_score'] = test_score
            #     print(f"{model_name} - Test F1 Score: {test_score:.4f}")
        
        return result
    
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 5)

print("Libraries loaded successfully")
print("Learning curve function defined")

Libraries loaded successfully
Learning curve function defined


## 1. Import Required Libraries

Import libraries for data handling, model training, cross-validation, and visualization.

# Learning Curves for Diabetes Prediction Models

**CS 1675 - Introduction to Machine Learning**

This notebook generates and analyzes learning curves for all three models (Logistic Regression, Neural Network, and Random Forest) to understand how they perform as a function of training set size.

Learning curves help us identify:
- **Variance issues**: Model performs well on training data but poorly on validation data (overfitting)
- **Bias issues**: Model performs poorly on both training and validation data (underfitting)
- **Sample efficiency**: How much data is needed for the model to reach good performance
- **Relative model quality**: Which model generalizes better with the same amount of data

---